# 03 — Model Training

Goals:
- Train LightGBM with default hyperparameters
- Run Optuna hyperparameter tuning (optional)
- Save the trained model to `backend/artifacts/`

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path
import json

from src.model.train import get_feature_cols, train_lgbm, save_model
from src.model.optimize import run_tuning, get_best_params

ARTIFACTS_DIR = Path('../backend/artifacts')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print('Ready.')

In [ ]:
# ── Load pre-computed feature parquet files (from notebook 02) ─────────────
train_df = pd.read_parquet('../data/train_features.parquet')
val_df   = pd.read_parquet('../data/val_features.parquet')
test_df  = pd.read_parquet('../data/test_features.parquet')

feat_cols = get_feature_cols(train_df)
X_train, y_train = train_df[feat_cols], train_df['y']
X_val,   y_val   = val_df[feat_cols],   val_df['y']
X_test,  y_test  = test_df[feat_cols],  test_df['y']

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'Features: {feat_cols}')

In [ ]:
# ── (Optional) Optuna hyperparameter tuning ────────────────────────────────
# Uncomment to run tuning. Takes ~5-20 min depending on n_trials.
# best_params = run_tuning(X_train, y_train, X_val, y_val,
#                          n_trials=50, artifacts_dir=ARTIFACTS_DIR)
# print('Best params:', best_params)

# Load pre-saved best params if available
try:
    best_params = get_best_params(ARTIFACTS_DIR)
    print('Loaded best_params.json:', best_params)
except FileNotFoundError:
    best_params = None
    print('No best_params.json found; using defaults.')

In [ ]:
# ── Train model ────────────────────────────────────────────────────────────
model, history = train_lgbm(X_train, y_train, X_val, y_val, params=best_params)
print('Training history:')
print(json.dumps({k: v for k, v in history.items() if k != 'params'}, indent=2))

In [ ]:
# ── Feature importance (LightGBM built-in) ─────────────────────────────────
import matplotlib.pyplot as plt
import lightgbm as lgb

fig, ax = plt.subplots(figsize=(10, 6))
lgb.plot_importance(model, ax=ax, max_num_features=20, importance_type='gain')
ax.set_title('Feature Importance (Gain)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Choose decision threshold ───────────────────────────────────────────────
from sklearn.metrics import precision_recall_curve

val_probs = model.predict(X_val)
precisions, recalls, thresholds = precision_recall_curve(y_val, val_probs)

f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]
print(f'Optimal threshold (max F1 on val): {best_threshold:.4f}')
print(f'Val F1 at threshold: {f1_scores[best_idx]:.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, precisions[:-1], label='Precision', color='#3b82f6')
ax.plot(thresholds, recalls[:-1],    label='Recall',    color='#f59e0b')
ax.plot(thresholds, f1_scores,       label='F1',        color='#22c55e')
ax.axvline(best_threshold, color='red', linestyle='--', label=f'Optimal threshold={best_threshold:.3f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs Threshold (Validation Set)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Save model ─────────────────────────────────────────────────────────────
from sklearn.metrics import average_precision_score

metrics = {'pr_auc_val': float(average_precision_score(y_val, val_probs))}
model_path = save_model(model, history.get('params', {}), metrics, ARTIFACTS_DIR, threshold=best_threshold)
print(f'Model saved to: {model_path}')